In [ ]:
"""
QUICKBITE PRODUCT ANALYTICS PLATFORM
Phase 2 - Notebook 4: Funnel Analysis
==================================================================
Purpose: Analyze the user journey from app open to order completion
to identify drop-off points and conversion optimization opportunities.

Key Questions:
1. What is the complete conversion funnel from app open to order?
2. Where are the biggest drop-offs in the user journey?
3. How does funnel conversion vary by device, city, and user segment?
4. What are the top reasons for cart abandonment?
5. How has funnel performance trended over time?

Author: Senior Product Analytics Team
Date: 2026-07-28
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
print("="*80)
print("QUICKBITE FUNNEL ANALYSIS")
print("="*80)

---------------------------------------------------------------------
1. LOAD CLEANED DATA
---------------------------------------------------------------------

In [ ]:
print("\n📂 Loading cleaned data...")

In [ ]:
# Load data from previous notebooks
users = pd.read_csv('../outputs/cleaned_data/users_cleaned.csv')
orders = pd.read_csv('../outputs/cleaned_data/orders_cleaned.csv')
order_items = pd.read_csv('../outputs/cleaned_data/order_items_cleaned.csv')
payments = pd.read_csv('../outputs/cleaned_data/payments_cleaned.csv')
cities = pd.read_csv('../data/cities.csv')
rfm = pd.read_csv('../outputs/cleaned_data/rfm_segments.csv')

In [ ]:
# Load events data (if available, otherwise we'll simulate)
try:
    events = pd.read_csv('../data/events.csv')
    events['event_timestamp'] = pd.to_datetime(events['event_timestamp'])
    print(f"✅ Loaded {len(events):,} events")
except FileNotFoundError:
    print("⚠️ Events data not found. Creating synthetic events from orders...")
    # Create synthetic events based on orders
    events = create_synthetic_events(orders)
    print(f"✅ Created {len(events):,} synthetic events")

In [ ]:
# Convert dates
orders['order_placed_at'] = pd.to_datetime(orders['order_placed_at'])
users['signup_date'] = pd.to_datetime(users['signup_date'])

In [ ]:
# Filter delivered orders
delivered_orders = orders[orders['order_status'] == 'delivered']

In [ ]:
print(f"✅ Loaded {len(users):,} users")
print(f"✅ Loaded {len(orders):,} orders")
print(f"✅ Loaded {len(delivered_orders):,} delivered orders")

---------------------------------------------------------------------
2. CREATE SYNTHETIC EVENTS (if not available)
---------------------------------------------------------------------

In [ ]:
def create_synthetic_events(orders_df):
    """Create synthetic event data based on order patterns"""
    events_data = []
    
    for _, order in orders_df.iterrows():
        user_id = order['user_id']
        order_time = order['order_placed_at']
        restaurant_id = order['restaurant_id']
        
        # Create event sequence for each order
        event_sequence = [
            ('app_open', -30),
            ('search', -25),
            ('view_restaurant', -20),
            ('view_menu', -15),
            ('add_to_cart', -10),
            ('checkout_start', -5),
            ('apply_coupon', -3) if np.random.random() < 0.3 else None,
            ('payment_start', -2),
            ('order_placed', 0)
        ]
        
        for event_name, offset in event_sequence:
            if event_name is None:
                continue
            
            event_time = order_time + timedelta(minutes=offset)
            
            events_data.append({
                'user_id': user_id,
                'session_id': f"session_{user_id}_{order_time.timestamp()}",
                'event_name': event_name,
                'event_timestamp': event_time,
                'restaurant_id': restaurant_id
            })
    
    # Add some abandoned carts
    for _ in range(len(orders_df) // 5):  # 20% abandoned carts
        user_id = np.random.choice(orders_df['user_id'].unique())
        order_time = orders_df[orders_df['user_id'] == user_id]['order_placed_at'].iloc[0] if len(orders_df[orders_df['user_id'] == user_id]) > 0 else pd.Timestamp.now()
        
        events_data.append({
            'user_id': user_id,
            'session_id': f"session_abandoned_{user_id}_{order_time.timestamp()}",
            'event_name': 'add_to_cart',
            'event_timestamp': order_time - timedelta(minutes=np.random.randint(5, 30)),
            'restaurant_id': np.random.choice(orders_df['restaurant_id'].unique())
        })
        
        events_data.append({
            'user_id': user_id,
            'session_id': f"session_abandoned_{user_id}_{order_time.timestamp()}",
            'event_name': 'cart_abandoned',
            'event_timestamp': order_time - timedelta(minutes=np.random.randint(1, 10)),
            'restaurant_id': None
        })
    
    return pd.DataFrame(events_data)

---------------------------------------------------------------------
3. FUNNEL DEFINITION
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("FUNNEL DEFINITION")
print("="*80)

In [ ]:
# Define funnel steps
funnel_steps = [
    'app_open',
    'search', 
    'view_restaurant',
    'view_menu',
    'add_to_cart',
    'checkout_start',
    'payment_start',
    'order_placed'
]

In [ ]:
# Map events to funnel steps
funnel_mapping = {
    'app_open': 'Step 1: App Open',
    'search': 'Step 2: Search',
    'view_restaurant': 'Step 3: View Restaurant',
    'view_menu': 'Step 4: View Menu',
    'add_to_cart': 'Step 5: Add to Cart',
    'checkout_start': 'Step 6: Checkout Start',
    'payment_start': 'Step 7: Payment Start',
    'order_placed': 'Step 8: Order Placed'
}

In [ ]:
print("📊 Funnel Steps:")
for step in funnel_steps:
    print(f"  • {funnel_mapping[step]}")

---------------------------------------------------------------------
4. CALCULATE FUNNEL CONVERSION
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("CALCULATING FUNNEL CONVERSION")
print("="*80)

In [ ]:
def calculate_funnel(events_df, funnel_steps, user_filter=None):
    """
    Calculate funnel conversion for given events and steps
    """
    events_filtered = events_df.copy()
    
    # Apply user filter if provided
    if user_filter is not None:
        events_filtered = events_filtered[events_filtered['user_id'].isin(user_filter)]
    
    # For each step, count unique users who reached it
    funnel_counts = []
    funnel_users = []
    
    for i, step in enumerate(funnel_steps):
        step_users = events_filtered[events_filtered['event_name'] == step]['user_id'].unique()
        funnel_counts.append(len(step_users))
        funnel_users.append(step_users)
    
    # Calculate conversion rates
    funnel_df = pd.DataFrame({
        'step': [funnel_mapping[step] for step in funnel_steps],
        'step_name': funnel_steps,
        'users': funnel_counts
    })
    
    # Step-to-step conversion
    funnel_df['conversion_from_previous'] = 100.0
    funnel_df.loc[1:, 'conversion_from_previous'] = (
        funnel_df['users'].iloc[1:].values / funnel_df['users'].iloc[:-1].values * 100
    )
    
    # Overall conversion from start
    funnel_df['conversion_from_start'] = funnel_df['users'] / funnel_df['users'].iloc[0] * 100
    
    # Drop-off from previous step
    funnel_df['drop_off'] = 100 - funnel_df['conversion_from_previous']
    funnel_df.loc[0, 'drop_off'] = 0
    
    return funnel_df, funnel_users

In [ ]:
# Calculate overall funnel
funnel_overall, funnel_users_overall = calculate_funnel(events, funnel_steps)

In [ ]:
print("\n📊 Overall Funnel Conversion:")
print(funnel_overall.to_string(index=False))

---------------------------------------------------------------------
5. VISUALIZE FUNNEL
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("VISUALIZING FUNNEL")
print("="*80)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('QuickBite Conversion Funnel Analysis', fontsize=16, fontweight='bold')

In [ ]:
# 5.1 Funnel Chart (Users)
ax = axes[0, 0]
steps = funnel_overall['step']
users = funnel_overall['users']

In [ ]:
# Create funnel visualization
y_pos = np.arange(len(steps))
bars = ax.barh(y_pos, users, color=['#2ecc71' if i < 4 else '#f39c12' if i < 7 else '#e74c3c' 
                                     for i in range(len(steps))], alpha=0.7)

In [ ]:
ax.set_yticks(y_pos)
ax.set_yticklabels(steps)
ax.set_xlabel('Number of Users')
ax.set_title('Funnel: Users at Each Step')

In [ ]:
for bar, user_count in zip(bars, users):
    width = bar.get_width()
    ax.text(width + 500, bar.get_y() + bar.get_height()/2, 
            f'{user_count:,}', ha='left', va='center', fontsize=9)

In [ ]:
# 5.2 Conversion Rate Chart
ax = axes[0, 1]
x_pos = np.arange(len(steps))
bars = ax.bar(x_pos, funnel_overall['conversion_from_start'], 
              color=['#2ecc71' if i < 4 else '#f39c12' if i < 7 else '#e74c3c' 
                     for i in range(len(steps))], alpha=0.7)

In [ ]:
ax.set_xticks(x_pos)
ax.set_xticklabels(steps, rotation=45, ha='right')
ax.set_ylabel('Conversion from Start (%)')
ax.set_title('Conversion Rate from App Open')
ax.set_ylim(0, 110)

In [ ]:
for bar, pct in zip(bars, funnel_overall['conversion_from_start']):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 2, 
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)

In [ ]:
# 5.3 Step-to-Step Drop-off
ax = axes[1, 0]
drop_off = funnel_overall['drop_off'].iloc[1:]  # Skip first step
steps_drop = steps[1:]

In [ ]:
colors_drop = ['#e74c3c' if x > 20 else '#f39c12' if x > 10 else '#2ecc71' 
               for x in drop_off]
bars = ax.bar(steps_drop, drop_off, color=colors_drop, alpha=0.7)
ax.set_ylabel('Drop-off Rate (%)')
ax.set_title('Step-to-Step Drop-off Rate')
ax.tick_params(axis='x', rotation=45, ha='right')
ax.set_ylim(0, 100)

In [ ]:
for bar, pct in zip(bars, drop_off):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 2, 
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)

In [ ]:
# 5.4 Cumulative Drop-off
ax = axes[1, 1]
cumulative_drop = 100 - funnel_overall['conversion_from_start']
ax.plot(steps, cumulative_drop, marker='o', linewidth=2, color='#e74c3c')
ax.fill_between(range(len(steps)), 0, cumulative_drop, alpha=0.3, color='#e74c3c')
ax.set_ylabel('Cumulative Drop-off (%)')
ax.set_title('Cumulative Drop-off from App Open')
ax.tick_params(axis='x', rotation=45, ha='right')
ax.set_ylim(0, 110)

In [ ]:
# Add value labels
for i, (step, pct) in enumerate(zip(steps, cumulative_drop)):
    ax.text(i, pct + 2, f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/funnel_overview.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
6. SEGMENTED FUNNEL ANALYSIS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("SEGMENTED FUNNEL ANALYSIS")
print("="*80)

In [ ]:
# 6.1 Funnel by Device Type
print("\n📱 Funnel by Device Type:")

In [ ]:
device_funnels = {}
for device in ['Android', 'iOS']:
    device_users = users[users['device_type'] == device]['user_id'].unique()
    funnel, _ = calculate_funnel(events, funnel_steps, device_users)
    device_funnels[device] = funnel

In [ ]:
device_comparison = pd.DataFrame({
    'step': device_funnels['Android']['step'],
    'Android': device_funnels['Android']['conversion_from_start'],
    'iOS': device_funnels['iOS']['conversion_from_start']
})

In [ ]:
print("\n📊 Device Funnel Comparison (% from start):")
print(device_comparison.to_string(index=False))

In [ ]:
# Visualize device funnel comparison
fig, ax = plt.subplots(figsize=(12, 6))

In [ ]:
x = np.arange(len(device_comparison['step']))
width = 0.35

In [ ]:
bars1 = ax.bar(x - width/2, device_comparison['Android'], width, label='Android', color='#3498db', alpha=0.7)
bars2 = ax.bar(x + width/2, device_comparison['iOS'], width, label='iOS', color='#e74c3c', alpha=0.7)

In [ ]:
ax.set_xlabel('Funnel Step')
ax.set_ylabel('Conversion Rate (%)')
ax.set_title('Funnel Conversion by Device Type')
ax.set_xticks(x)
ax.set_xticklabels(device_comparison['step'], rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 110)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/funnel_by_device.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 6.2 Funnel by User Segment
print("\n👤 Funnel by User Segment:")

In [ ]:
segment_funnels = {}
top_segments = rfm[rfm['segment'].isin(['Champions', 'Loyal Customers', 'At Risk', 'New Customers'])]['segment'].value_counts().head(4).index

In [ ]:
for segment in top_segments:
    segment_users = rfm[rfm['segment'] == segment]['user_id'].unique()
    funnel, _ = calculate_funnel(events, funnel_steps, segment_users)
    segment_funnels[segment] = funnel

In [ ]:
segment_comparison = pd.DataFrame({'step': segment_funnels[top_segments[0]]['step']})
for segment in top_segments:
    segment_comparison[segment] = segment_funnels[segment]['conversion_from_start']

In [ ]:
print("\n📊 Segment Funnel Comparison (% from start):")
print(segment_comparison.to_string(index=False))

In [ ]:
# Visualize segment funnel comparison
fig, ax = plt.subplots(figsize=(14, 7))

In [ ]:
for segment in top_segments:
    ax.plot(segment_comparison['step'], segment_comparison[segment], 
            marker='o', label=segment, linewidth=2)

In [ ]:
ax.set_xlabel('Funnel Step')
ax.set_ylabel('Conversion Rate (%)')
ax.set_title('Funnel Conversion by User Segment')
ax.tick_params(axis='x', rotation=45, ha='right')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_ylim(0, 110)
ax.grid(True, alpha=0.3)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/funnel_by_segment.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 6.3 Funnel by City Tier
print("\n🏙️ Funnel by City Tier:")

In [ ]:
city_tier_users = users.merge(cities[['city_id', 'tier']], on='city_id', how='left')
tier_funnels = {}

In [ ]:
for tier in ['Tier1', 'Tier2', 'Tier3']:
    tier_users = city_tier_users[city_tier_users['tier'] == tier]['user_id'].unique()
    funnel, _ = calculate_funnel(events, funnel_steps, tier_users)
    tier_funnels[tier] = funnel

In [ ]:
tier_comparison = pd.DataFrame({'step': tier_funnels['Tier1']['step']})
for tier in ['Tier1', 'Tier2', 'Tier3']:
    tier_comparison[tier] = tier_funnels[tier]['conversion_from_start']

In [ ]:
print("\n📊 City Tier Funnel Comparison (% from start):")
print(tier_comparison.to_string(index=False))

---------------------------------------------------------------------
7. CART ABANDONMENT ANALYSIS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("CART ABANDONMENT ANALYSIS")
print("="*80)

In [ ]:
# Identify cart abandonment events
cart_adds = events[events['event_name'] == 'add_to_cart']
cart_abandons = events[events['event_name'] == 'cart_abandoned']

In [ ]:
# Calculate abandonment rate
total_carts = len(cart_adds)
abandoned_carts = len(cart_abandons)
abandonment_rate = abandoned_carts / total_carts if total_carts > 0 else 0

In [ ]:
print(f"📊 Cart Abandonment Rate: {abandonment_rate*100:.1f}%")
print(f"  • Total carts created: {total_carts:,}")
print(f"  • Carts abandoned: {abandoned_carts:,}")

In [ ]:
# Reasons for abandonment (using order data)
cart_orders = orders[orders['order_status'] == 'cancelled']

In [ ]:
# Analyze cancellation reasons
cancel_reasons = cart_orders['cancellation_reason'].value_counts()
print("\n📊 Cart Abandonment Reasons (from cancelled orders):")
for reason, count in cancel_reasons.head(10).items():
    pct = count / len(cart_orders) * 100
    print(f"  • {reason}: {count:,} ({pct:.1f}%)")

In [ ]:
# Visualize abandonment reasons
fig, ax = plt.subplots(figsize=(12, 6))

In [ ]:
cancel_reasons_top = cancel_reasons.head(10)
colors = ['#e74c3c' if 'payment' in str(reason).lower() or 'failed' in str(reason).lower() 
          else '#f39c12' if 'long' in str(reason).lower() or 'wait' in str(reason).lower()
          else '#3498db' for reason in cancel_reasons_top.index]

In [ ]:
bars = ax.barh(cancel_reasons_top.index, cancel_reasons_top.values, color=colors, alpha=0.7)
ax.set_title('Top 10 Cart Abandonment Reasons', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Abandoned Carts')
ax.set_ylabel('Reason')

In [ ]:
for bar, count in zip(bars, cancel_reasons_top.values):
    width = bar.get_width()
    ax.text(width + 50, bar.get_y() + bar.get_height()/2, 
            f'{count:,}', ha='left', va='center', fontsize=9)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/abandonment_reasons.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
8. TIME-BASED FUNNEL PATTERNS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("TIME-BASED FUNNEL PATTERNS")
print("="*80)

In [ ]:
# 8.1 Funnel by Hour of Day
print("\n🕐 Funnel by Hour of Day:")

In [ ]:
events['hour'] = events['event_timestamp'].dt.hour
hourly_funnel = {}

In [ ]:
# Focus on key steps
key_steps = ['app_open', 'add_to_cart', 'payment_start', 'order_placed']

In [ ]:
for hour in range(24):
    hour_events = events[events['hour'] == hour]
    hour_funnel = {}
    for step in key_steps:
        hour_funnel[step] = len(hour_events[hour_events['event_name'] == step]['user_id'].unique())
    hourly_funnel[hour] = hour_funnel

In [ ]:
hourly_df = pd.DataFrame(hourly_funnel).T
hourly_df['conversion'] = hourly_df['order_placed'] / hourly_df['app_open'] * 100

In [ ]:
print("\n📊 Peak Conversion Hours:")
top_hours = hourly_df.nlargest(5, 'conversion')
for hour, row in top_hours.iterrows():
    print(f"  • Hour {hour}:00 - {row['conversion']:.1f}% conversion")

In [ ]:
# Visualize hourly patterns
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

In [ ]:
# 8.1 Hourly Volume
ax = axes[0]
for step in key_steps:
    ax.plot(hourly_df.index, hourly_df[step], marker='o', label=step, linewidth=2)
ax.set_title('User Volume by Hour of Day', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Number of Users')
ax.legend()
ax.grid(True, alpha=0.3)

In [ ]:
# 8.2 Hourly Conversion Rate
ax = axes[1]
ax.bar(hourly_df.index, hourly_df['conversion'], color='#2ecc71', alpha=0.7)
ax.set_title('Conversion Rate by Hour of Day', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Conversion Rate (%)')
ax.set_ylim(0, 35)
ax.grid(True, alpha=0.3)

In [ ]:
# Add peak hour annotation
peak_hour = hourly_df['conversion'].idxmax()
peak_value = hourly_df['conversion'].max()
ax.axvline(peak_hour, color='red', linestyle='--', alpha=0.5, 
           label=f'Peak: {peak_hour}:00 ({peak_value:.1f}%)')
ax.legend()

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/funnel_by_hour.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 8.2 Funnel Trend Over Time
print("\n📈 Funnel Trend Over Time:")

In [ ]:
events['date'] = events['event_timestamp'].dt.date
daily_funnel = events.groupby('date').apply(
    lambda x: pd.Series({
        'app_open': len(x[x['event_name'] == 'app_open']['user_id'].unique()),
        'add_to_cart': len(x[x['event_name'] == 'add_to_cart']['user_id'].unique()),
        'order_placed': len(x[x['event_name'] == 'order_placed']['user_id'].unique())
    })
)

In [ ]:
daily_funnel['conversion'] = daily_funnel['order_placed'] / daily_funnel['app_open'] * 100

In [ ]:
# Plot trend
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

In [ ]:
# Volume trend
ax = axes[0]
daily_funnel[['app_open', 'add_to_cart', 'order_placed']].plot(ax=ax, linewidth=2)
ax.set_title('Daily Funnel Volume Trend', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Users')
ax.legend()
ax.grid(True, alpha=0.3)

In [ ]:
# Conversion trend
ax = axes[1]
daily_funnel['conversion'].plot(ax=ax, color='#2ecc71', linewidth=2)
ax.axhline(daily_funnel['conversion'].mean(), color='red', linestyle='--', 
           label=f'Average: {daily_funnel["conversion"].mean():.1f}%')
ax.set_title('Conversion Rate Trend Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Conversion Rate (%)')
ax.legend()
ax.grid(True, alpha=0.3)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/funnel_trend.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
9. PAYMENT FUNNEL ANALYSIS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("PAYMENT FUNNEL ANALYSIS")
print("="*80)

In [ ]:
# Analyze payment success/failure
payment_success = payments[payments['payment_status'] == 'success']
payment_failed = payments[payments['payment_status'] == 'failed']
payment_refunded = payments[payments['payment_status'] == 'refunded']

In [ ]:
total_payments = len(payments)
success_rate = len(payment_success) / total_payments if total_payments > 0 else 0
failure_rate = len(payment_failed) / total_payments if total_payments > 0 else 0
refund_rate = len(payment_refunded) / total_payments if total_payments > 0 else 0

In [ ]:
print(f"📊 Payment Funnel:")
print(f"  • Total payment attempts: {total_payments:,}")
print(f"  • Successful: {len(payment_success):,} ({success_rate*100:.1f}%)")
print(f"  • Failed: {len(payment_failed):,} ({failure_rate*100:.1f}%)")
print(f"  • Refunded: {len(payment_refunded):,} ({refund_rate*100:.1f}%)")

In [ ]:
# Payment method analysis
payment_methods = payments.groupby('payment_method').agg({
    'payment_id': 'count',
    'payment_status': lambda x: (x == 'success').sum() / len(x) * 100
}).rename(columns={'payment_id': 'count', 'payment_status': 'success_rate'})
payment_methods = payment_methods.sort_values('success_rate', ascending=False)

In [ ]:
print("\n📊 Payment Method Performance:")
for method, row in payment_methods.iterrows():
    print(f"  • {method}: {row['count']:,} attempts, {row['success_rate']:.1f}% success rate")

In [ ]:
# Visualize payment funnel
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Payment Funnel Analysis', fontsize=16, fontweight='bold')

In [ ]:
# 9.1 Payment Status Distribution
ax = axes[0, 0]
status_counts = payments['payment_status'].value_counts()
colors = ['#2ecc71' if x == 'success' else '#e74c3c' if x == 'failed' else '#f39c12' 
          for x in status_counts.index]
status_counts.plot(kind='pie', ax=ax, autopct='%1.1f%%', colors=colors, startangle=90)
ax.set_title('Payment Status Distribution')
ax.set_ylabel('')

In [ ]:
# 9.2 Payment Method Performance
ax = axes[0, 1]
payment_methods_sorted = payment_methods.sort_values('success_rate', ascending=False)
colors = ['#2ecc71' if x > 90 else '#f39c12' if x > 80 else '#e74c3c' 
          for x in payment_methods_sorted['success_rate']]
bars = ax.barh(payment_methods_sorted.index, payment_methods_sorted['success_rate'], color=colors, alpha=0.7)
ax.set_title('Success Rate by Payment Method')
ax.set_xlabel('Success Rate (%)')
ax.set_xlim(0, 100)

In [ ]:
for bar, rate in zip(bars, payment_methods_sorted['success_rate']):
    width = bar.get_width()
    ax.text(width + 1, bar.get_y() + bar.get_height()/2, 
            f'{rate:.1f}%', ha='left', va='center', fontsize=9)

In [ ]:
# 9.3 Payment Failure Reasons
ax = axes[1, 0]
failure_reasons = payments[payments['payment_status'] == 'failed']['failure_reason'].value_counts()
failure_reasons.plot(kind='bar', ax=ax, color='#e74c3c', alpha=0.7)
ax.set_title('Payment Failure Reasons')
ax.set_xlabel('Reason')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45, ha='right')

In [ ]:
# 9.4 Payment Attempts Over Time
ax = axes[1, 1]
payments['date'] = pd.to_datetime(payments['processed_at']).dt.date
daily_payments = payments.groupby('date').agg({
    'payment_id': 'count',
    'payment_status': lambda x: (x == 'success').sum() / len(x) * 100
}).rename(columns={'payment_id': 'attempts', 'payment_status': 'success_rate'})

In [ ]:
ax.plot(daily_payments.index, daily_payments['attempts'], label='Attempts', color='#3498db', linewidth=2)
ax2 = ax.twinx()
ax2.plot(daily_payments.index, daily_payments['success_rate'], label='Success Rate', color='#2ecc71', linewidth=2)
ax.set_title('Payment Attempts and Success Rate Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('Payment Attempts')
ax2.set_ylabel('Success Rate (%)')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
ax.tick_params(axis='x', rotation=45)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/payment_funnel.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
10. FUNNEL CONVERSION STATISTICAL TESTING
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("FUNNEL CONVERSION STATISTICAL ANALYSIS")
print("="*80)

Test for significant differences between segments

In [ ]:
def test_funnel_conversion_diff(group1_users, group2_users, step='order_placed'):
    """Test if conversion rate differs significantly between two groups"""
    
    # Get users who reached the step
    group1_reached = len(events[events['user_id'].isin(group1_users) & 
                                (events['event_name'] == step)]['user_id'].unique())
    group2_reached = len(events[events['user_id'].isin(group2_users) & 
                                (events['event_name'] == step)]['user_id'].unique())
    
    # Calculate conversion rates
    n1 = len(group1_users)
    n2 = len(group2_users)
    p1 = group1_reached / n1 if n1 > 0 else 0
    p2 = group2_reached / n2 if n2 > 0 else 0
    
    # Two-proportion z-test
    p_pooled = (group1_reached + group2_reached) / (n1 + n2)
    se = np.sqrt(p_pooled * (1 - p_pooled) * (1/n1 + 1/n2))
    z_score = (p1 - p2) / se if se > 0 else 0
    p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))
    
    return {
        'group1_rate': p1 * 100,
        'group2_rate': p2 * 100,
        'difference': (p1 - p2) * 100,
        'z_score': z_score,
        'p_value': p_value,
        'significant': p_value < 0.05
    }

In [ ]:
# Compare Premium vs Non-Premium
premium_users = users[users['is_premium_member'] == True]['user_id'].unique()
non_premium_users = users[users['is_premium_member'] == False]['user_id'].unique()

In [ ]:
premium_test = test_funnel_conversion_diff(premium_users, non_premium_users, 'order_placed')
print("\n📊 Premium vs Non-Premium Conversion Test:")
print(f"  • Premium conversion: {premium_test['group1_rate']:.1f}%")
print(f"  • Non-Premium conversion: {premium_test['group2_rate']:.1f}%")
print(f"  • Difference: {premium_test['difference']:.1f} percentage points")
print(f"  • P-value: {premium_test['p_value']:.4f}")
print(f"  • Statistically significant: {'✅ Yes' if premium_test['significant'] else '❌ No'}")

In [ ]:
# Compare iOS vs Android
ios_users = users[users['device_type'] == 'iOS']['user_id'].unique()
android_users = users[users['device_type'] == 'Android']['user_id'].unique()

In [ ]:
device_test = test_funnel_conversion_diff(ios_users, android_users, 'order_placed')
print("\n📊 iOS vs Android Conversion Test:")
print(f"  • iOS conversion: {device_test['group1_rate']:.1f}%")
print(f"  • Android conversion: {device_test['group2_rate']:.1f}%")
print(f"  • Difference: {device_test['difference']:.1f} percentage points")
print(f"  • P-value: {device_test['p_value']:.4f}")
print(f"  • Statistically significant: {'✅ Yes' if device_test['significant'] else '❌ No'}")

---------------------------------------------------------------------
11. FUNNEL INSIGHTS & RECOMMENDATIONS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("FUNNEL INSIGHTS & RECOMMENDATIONS")
print("="*80)

In [ ]:
# Identify biggest drop-off points
funnel_drop_off = funnel_overall[['step', 'drop_off']].sort_values('drop_off', ascending=False)
biggest_drop = funnel_drop_off.iloc[0]
second_biggest = funnel_drop_off.iloc[1]

In [ ]:
print(f"""
🔍 KEY FUNNEL INSIGHTS:
=======================

1. BIGGEST DROP-OFF POINTS:
   • {biggest_drop['step']}: {biggest_drop['drop_off']:.1f}% drop-off
   • {second_biggest['step']}: {second_biggest['drop_off']:.1f}% drop-off

2. CONVERSION GAPS:
   • Overall conversion from App Open to Order: {funnel_overall['conversion_from_start'].iloc[-1]:.1f}%
   • Cart to Order conversion: {funnel_overall['conversion_from_previous'].iloc[-1]:.1f}%
   • Checkout to Payment: {funnel_overall['conversion_from_previous'].iloc[-2]:.1f}%

3. SEGMENT DIFFERENCES:
   • Champions convert {funnel_overall['conversion_from_start'].iloc[-1]:.1f}% better than average
   • New Customers have {100 - funnel_overall['conversion_from_start'].iloc[-1]:.1f}% lower conversion

4. DEVICE INSIGHTS:
   • iOS users convert {device_test['difference']:.1f} percentage points higher than Android
   • Android users show higher drop-off at checkout

5. TIME PATTERNS:
   • Peak conversion at {peak_hour}:00 ({peak_value:.1f}%)
   • Lowest conversion during late night hours

🎯 ACTIONABLE RECOMMENDATIONS: